In [4]:
import numpy as np
import scipy
import pandas as pd
import math
import random
import sklearn
from nltk.corpus import stopwords
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [5]:
articles_df = pd.read_csv('shared_articles.csv')
articles_df = articles_df[articles_df['eventType'] == 'CONTENT SHARED']
articles_df.head(5)

,timestamp,eventType,contentId,authorPersonId,authorSessionId,authorUserAgent,authorRegion,authorCountry,contentType,url,title,text,lang
1,1459193988,CONTENT SHARED,-4110354420726924665,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.nytimes.com/2016/03/28/business/dea...,"Ethereum, a Virtual Currency, Enables Transact...",All of this work is still very early. The firs...,en
2,1459194146,CONTENT SHARED,-7292285110016212249,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://cointelegraph.com/news/bitcoin-future-w...,Bitcoin Future: When GBPcoin of Branson Wins O...,The alarm clock wakes me at 8:00 with stream o...,en
3,1459194474,CONTENT SHARED,-6151852268067518688,3891637997717104548,-1457532940883382585,NaN,NaN,NaN,HTML,https://cloudplatform.googleblog.com/2016/03/G...,Google Data Center 360° Tour,We're excited to share the Google Data Center ...,en
4,1459194497,CONTENT SHARED,2448026894306402386,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,https://bitcoinmagazine.com/articles/ibm-wants...,"IBM Wants to ""Evolve the Internet"" With Blockc...",The Aite Group projects the blockchain market ...,en
5,1459194522,CONTENT SHARED,-2826566343807132236,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.coindesk.com/ieee-blockchain-oxford...,IEEE to Talk Blockchain at Cloud Computing Oxf...,One of the largest and oldest organizations fo...,en


In [6]:
interactions_df = pd.read_csv('users_interactions.csv')
interactions_df.head(10)

,timestamp,eventType,contentId,personId,sessionId,userAgent,userRegion,userCountry
0,1465413032,VIEW,-3499919498720038879,-8845298781299428018,1264196770339959068,NaN,NaN,NaN
1,1465412560,VIEW,8890720798209849691,-1032019229384696495,3621737643587579081,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,NY,US
2,1465416190,VIEW,310515487419366995,-1130272294246983140,2631864456530402479,NaN,NaN,NaN
3,1465413895,FOLLOW,310515487419366995,344280948527967603,-3167637573980064150,NaN,NaN,NaN
4,1465412290,VIEW,-7820640624231356730,-445337111692715325,5611481178424124714,NaN,NaN,NaN
5,1465413742,VIEW,310515487419366995,-8763398617720485024,1395789369402380392,Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebK...,MG,BR
6,1465415950,VIEW,-8864073373672512525,3609194402293569455,1143207167886864524,NaN,NaN,NaN
7,1465415066,VIEW,-1492913151930215984,4254153380739593270,8743229464706506141,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,SP,BR
8,1465413762,VIEW,310515487419366995,344280948527967603,-3167637573980064150,NaN,NaN,NaN
9,1465413771,VIEW,3064370296170038610,3609194402293569455,1143207167886864524,NaN,NaN,NaN


In [7]:
interactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72312 entries, 0 to 72311
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   timestamp    72312 non-null  int64 
 1   eventType    72312 non-null  object
 2   contentId    72312 non-null  int64 
 3   personId     72312 non-null  int64 
 4   sessionId    72312 non-null  int64 
 5   userAgent    56918 non-null  object
 6   userRegion   56907 non-null  object
 7   userCountry  56918 non-null  object
dtypes: int64(4), object(4)
memory usage: 4.4+ MB


In [8]:
event_type_strength = {
   'VIEW': 1.0,
   'LIKE': 2.0,
   'BOOKMARK': 2.5,
   'FOLLOW': 3.0,
   'COMMENT CREATED': 4.0,
}

interactions_df['eventStrength'] = interactions_df['eventType'].apply(lambda x: event_type_strength[x])

In [9]:
users_interactions_count_df = interactions_df.groupby(['personId', 'contentId']).size().groupby('personId').size()
print('# users: %d' % len(users_interactions_count_df))
users_with_enough_interactions_df = users_interactions_count_df[users_interactions_count_df >= 5].reset_index()[['personId']]
print('# users with at least 5 interactions: %d' % len(users_with_enough_interactions_df))

# users: 1895
# users with at least 5 interactions: 1140


In [10]:
print('# of interactions: %d' % len(interactions_df))
interactions_from_selected_users_df = interactions_df.merge(users_with_enough_interactions_df,
               how = 'right',
               left_on = 'personId',
               right_on = 'personId')
print('# of interactions from users with at least 5 interactions: %d' % len(interactions_from_selected_users_df))

# of interactions: 72312
# of interactions from users with at least 5 interactions: 69868


In [11]:
def smooth_user_preference(x):
    return math.log(1+x, 2)

interactions_full_df = interactions_from_selected_users_df \
                    .groupby(['personId', 'contentId'])['eventStrength'].sum() \
                    .apply(smooth_user_preference).reset_index()
print('# of unique user/item interactions: %d' % len(interactions_full_df))
interactions_full_df.head(10)

# of unique user/item interactions: 39106


,personId,contentId,eventStrength
0,-9223121837663643404,-8949113594875411859,1.000000
1,-9223121837663643404,-8377626164558006982,1.000000
2,-9223121837663643404,-8208801367848627943,1.000000
3,-9223121837663643404,-8187220755213888616,1.000000
4,-9223121837663643404,-7423191370472335463,3.169925
5,-9223121837663643404,-7331393944609614247,1.000000
6,-9223121837663643404,-6872546942144599345,1.000000
7,-9223121837663643404,-6728844082024523434,1.000000
8,-9223121837663643404,-6590819806697898649,1.000000
9,-9223121837663643404,-6558712014192834002,1.584963


In [12]:
interactions_train_df, interactions_test_df = train_test_split(interactions_full_df,
                                   stratify=interactions_full_df['personId'],
                                   test_size=0.20,
                                   random_state=42)

print('# interactions on Train set: %d' % len(interactions_train_df))
print('# interactions on Test set: %d' % len(interactions_test_df))

# interactions on Train set: 31284
# interactions on Test set: 7822


In [13]:
interactions_full_indexed_df = interactions_full_df.set_index('personId')
interactions_train_indexed_df = interactions_train_df.set_index('personId')
interactions_test_indexed_df = interactions_test_df.set_index('personId')

In [14]:
def get_items_interacted(person_id, interactions_df):
    interacted_items = interactions_df.loc[person_id]['contentId']
    return set(interacted_items if type(interacted_items) == pd.Series else [interacted_items])

In [30]:
EVAL_RANDOM_SAMPLE_NON_INTERACTED_ITEMS = 100

class ModelEvaluator:

    def get_not_interacted_items_sample(self, person_id, sample_size, seed=42):
        interacted_items = get_items_interacted(person_id, interactions_full_indexed_df)
        all_items = set(articles_df['contentId'])
        non_interacted_items = all_items - interacted_items

        if not non_interacted_items:
            return set()

        random.seed(seed)
        actual_sample_size = min(sample_size, len(non_interacted_items))
        non_interacted_items_sample = random.sample(list(non_interacted_items), actual_sample_size)
        return set(non_interacted_items_sample)

    def _verify_hit_top_n(self, item_id, recommended_items, topn):
        try:
            index = next(i for i, c in enumerate(recommended_items) if c == item_id)
        except StopIteration:
            index = -1
        hit = int(index in range(0, topn))
        return hit, index

    def _compute_ndcg_at_topn(self, recommended_items, relevant_item_id, topn=10):
        dcg = 0.0
        for i, item in enumerate(recommended_items[:topn]):
            if item == relevant_item_id:
                dcg += 1.0 / math.log2(i + 2)
        idcg = 1.0 / math.log2(2)
        if idcg == 0:
            return 0.0
        return dcg / idcg

    def evaluate_model_for_user(self, model, person_id):
        interacted_values_testset = interactions_test_indexed_df.loc[person_id]
        if type(interacted_values_testset['contentId']) == pd.Series:
            person_interacted_items_testset = set(interacted_values_testset['contentId'])
        else:
            person_interacted_items_testset = {int(interacted_values_testset['contentId'])}
        interacted_items_count_testset = len(person_interacted_items_testset)

        person_recs_df = model.recommend_items(person_id,
                                               items_to_ignore=get_items_interacted(person_id,
                                                                                    interactions_train_indexed_df),
                                               topn=10000000000)

        hits_at_5_count = 0
        hits_at_10_count = 0
        ndcg_sum = 0.0

        for item_id in person_interacted_items_testset:
            non_interacted_items_sample = self.get_not_interacted_items_sample(person_id,
                                                                              sample_size=EVAL_RANDOM_SAMPLE_NON_INTERACTED_ITEMS,
                                                                              seed=item_id % (2 ** 32))

            items_to_filter_recs = non_interacted_items_sample.union({item_id})

            valid_recs_df = person_recs_df[person_recs_df['contentId'].isin(items_to_filter_recs)]
            valid_recs = valid_recs_df['contentId'].values
            hit_at_5, _ = self._verify_hit_top_n(item_id, valid_recs, 5)
            hits_at_5_count += hit_at_5
            hit_at_10, _ = self._verify_hit_top_n(item_id, valid_recs, 10)
            hits_at_10_count += hit_at_10

            ndcg = self._compute_ndcg_at_topn(valid_recs, item_id, topn=10)
            ndcg_sum += ndcg

        recall_at_5 = hits_at_5_count / float(interacted_items_count_testset)
        recall_at_10 = hits_at_10_count / float(interacted_items_count_testset)
        ndcg_at_10 = ndcg_sum / float(interacted_items_count_testset)

        person_metrics = {'hits@5_count': hits_at_5_count,
                          'hits@10_count': hits_at_10_count,
                          'interacted_count': interacted_items_count_testset,
                          'recall@5': recall_at_5,
                          'recall@10': recall_at_10,
                          'ndcg@10': ndcg_at_10}
        return person_metrics

    def evaluate_model(self, model):
        people_metrics = []
        user_ids = list(interactions_test_indexed_df.index.unique().values)
        for idx, person_id in enumerate(user_ids):
            person_metrics = self.evaluate_model_for_user(model, person_id)
            person_metrics['_person_id'] = person_id
            people_metrics.append(person_metrics)
        print(f'{len(people_metrics)} users processed')

        detailed_results_df = pd.DataFrame(people_metrics) \
                                .sort_values('interacted_count', ascending=False)

        global_recall_at_5 = detailed_results_df['hits@5_count'].sum() / float(detailed_results_df['interacted_count'].sum())
        global_recall_at_10 = detailed_results_df['hits@10_count'].sum() / float(detailed_results_df['interacted_count'].sum())
        total_ndcg_weighted = (detailed_results_df['ndcg@10'] * detailed_results_df['interacted_count']).sum()
        global_ndcg_at_10 = total_ndcg_weighted / detailed_results_df['interacted_count'].sum()


        global_metrics = {'modelName': model.get_model_name(),
                          'recall@5': global_recall_at_5,
                          'recall@10': global_recall_at_10,
                          'ndcg@10': global_ndcg_at_10}

        return global_metrics, detailed_results_df

model_evaluator = ModelEvaluator()

In [31]:
item_popularity_df = interactions_full_df.groupby('contentId')['eventStrength'].sum().sort_values(ascending=False).reset_index()
item_popularity_df.head(10)

,contentId,eventStrength
0,-4029704725707465084,307.733799
1,-6783772548752091658,233.762157
2,-133139342397538859,228.024567
3,-8208801367848627943,197.107608
4,-6843047699859121724,193.825208
5,8224860111193157980,189.044680
6,-2358756719610361882,183.110951
7,2581138407738454418,180.282876
8,7507067965574797372,179.094002
9,1469580151036142903,170.548969


In [32]:
class PopularityRecommender:

    MODEL_NAME = 'Popularity'

    def __init__(self, popularity_df, items_df=None):
        self.popularity_df = popularity_df
        self.items_df = items_df

    def get_model_name(self):
        return self.MODEL_NAME

    def recommend_items(self, user_id, items_to_ignore=[], topn=10, verbose=False):
        recommendations_df = self.popularity_df[~self.popularity_df['contentId'].isin(items_to_ignore)] \
                               .sort_values('eventStrength', ascending = False) \
                               .head(topn)

        if verbose:
            if self.items_df is None:
                raise Exception('"items_df" is required in verbose mode')

            recommendations_df = recommendations_df.merge(self.items_df, how = 'left',
                                                          left_on = 'contentId',
                                                          right_on = 'contentId')[['eventStrength', 'contentId', 'title', 'url', 'lang']]


        return recommendations_df

popularity_model = PopularityRecommender(item_popularity_df, articles_df)

In [33]:
print('Evaluating Popularity recommendation model...')
pop_global_metrics, pop_detailed_results_df = model_evaluator.evaluate_model(popularity_model)
print('\nGlobal metrics:\n%s' % pop_global_metrics)
pop_detailed_results_df.head(10)

Evaluating Popularity recommendation model...
1140 users processed

Global metrics:
{'modelName': 'Popularity', 'recall@5': np.float64(0.2418818716440808), 'recall@10': np.float64(0.3725389925850166), 'ndcg@10': np.float64(0.20234484668618488)}


,hits@5_count,hits@10_count,interacted_count,recall@5,recall@10,ndcg@10,_person_id
76,28,50,192,0.145833,0.260417,0.121697,3609194402293569455
17,12,25,134,0.089552,0.186567,0.082108,-2626634673110551643
16,13,23,130,0.100000,0.176923,0.079155,-1032019229384696495
10,5,9,117,0.042735,0.076923,0.044380,-1443636648652872475
82,26,40,88,0.295455,0.454545,0.243441,-2979881261169775358
161,12,18,80,0.150000,0.225000,0.124948,-3596626804281480007
65,20,34,73,0.273973,0.465753,0.237265,1116121227607581999
106,14,18,69,0.202899,0.260870,0.146918,-9016528795238256703
81,17,23,69,0.246377,0.333333,0.174707,692689608292948411
52,21,28,68,0.308824,0.411765,0.234381,3636910968448833585


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from nltk.corpus import stopwords
import nltk

In [35]:
nltk.download('stopwords')
stop_words = stopwords.words('english')

articles_df['combined_text'] = articles_df['title'].fillna('') + ' ' + articles_df['text'].fillna('')

tfidf = TfidfVectorizer(max_features=5000, stop_words=stop_words)
tfidf_matrix = tfidf.fit_transform(articles_df['combined_text'])

svd = TruncatedSVD(n_components=100, random_state=42)
item_embeddings = svd.fit_transform(tfidf_matrix)

item_embeddings = normalize(item_embeddings)

embedding_dict = dict(zip(articles_df['contentId'], item_embeddings))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
class ContentBasedRecommender:
    MODEL_NAME = 'ContentBased'

    def __init__(self, items_df, interactions_train_df, embedding_dict):
        self.items_df = items_df
        self.interactions_train_df = interactions_train_df
        self.embedding_dict = embedding_dict
        self._user_profile_cache = {}

    def get_model_name(self):
        return self.MODEL_NAME

    def _get_user_profile(self, user_id):
        if user_id in self._user_profile_cache:
            return self._user_profile_cache[user_id]

        user_items = self.interactions_train_df[
            self.interactions_train_df['personId'] == user_id
        ]['contentId'].values

        embeddings = []
        for item in user_items:
            if item in self.embedding_dict:
                embeddings.append(self.embedding_dict[item])

        if not embeddings:
            profile = None
        else:
            profile = np.mean(embeddings, axis=0)
            profile = profile / np.linalg.norm(profile)

        self._user_profile_cache[user_id] = profile
        return profile

    def recommend_items(self, user_id, items_to_ignore=None, topn=10, verbose=False):
        if items_to_ignore is None:
            items_to_ignore = []

        profile = self._get_user_profile(user_id)
        if profile is None:
            return pd.DataFrame(columns=['contentId'])

        all_items = set(self.embedding_dict.keys())
        candidate_items = all_items - set(items_to_ignore)
        if not candidate_items:
            return pd.DataFrame(columns=['contentId'])

        similarities = []
        for item_id in candidate_items:
            emb = self.embedding_dict[item_id]
            sim = np.dot(profile, emb)
            similarities.append((item_id, sim))

        similarities.sort(key=lambda x: x[1], reverse=True)
        top_items = similarities[:topn]

        recs_df = pd.DataFrame(top_items, columns=['contentId', 'similarity'])

        if verbose:
            recs_df = recs_df.merge(
                self.items_df[['contentId', 'title', 'url', 'lang']],
                on='contentId', how='left'
            )

        return recs_df

In [37]:
content_model = ContentBasedRecommender(articles_df, interactions_train_df, embedding_dict)

print('Evaluating Content‑Based recommendation model...')
cb_global_metrics, cb_detailed_results_df = model_evaluator.evaluate_model(content_model)
print('\nGlobal metrics:\n%s' % cb_global_metrics)

Evaluating Content‑Based recommendation model...
1140 users processed

Global metrics:
{'modelName': 'ContentBased', 'recall@5': np.float64(0.12426489388903093), 'recall@10': np.float64(0.2027614420864229), 'ndcg@10': np.float64(0.1062654621552537)}


In [38]:
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.linalg import svds
import numpy as np
import pandas as pd

class ALSRecommender:
    MODEL_NAME = 'ALS (SVD)'

    def __init__(self, interactions_train_df, items_df=None, n_factors=50, n_iter=10, alpha=1.0):
        self.interactions_train_df = interactions_train_df
        self.items_df = items_df
        self.n_factors = n_factors

        self.user_ids = interactions_train_df['personId'].unique()
        self.item_ids = interactions_train_df['contentId'].unique()
        self.user_to_idx = {uid: i for i, uid in enumerate(self.user_ids)}
        self.item_to_idx = {iid: j for j, iid in enumerate(self.item_ids)}
        self.idx_to_item = {j: iid for iid, j in self.item_to_idx.items()}

        rows = [self.user_to_idx[uid] for uid in interactions_train_df['personId']]
        cols = [self.item_to_idx[iid] for iid in interactions_train_df['contentId']]
        data = interactions_train_df['eventStrength'].values
        self.user_item_matrix = coo_matrix((data, (rows, cols)), shape=(len(self.user_ids), len(self.item_ids))).tocsr()

        U, sigma, Vt = svds(self.user_item_matrix, k=self.n_factors, which='LM')
        sigma = np.diag(sigma)
        self.user_factors = U @ sigma
        self.item_factors = Vt.T

        self._prediction_cache = {}

        item_pop = interactions_train_df.groupby('contentId')['eventStrength'].sum().sort_values(ascending=False)
        self.popular_items = item_pop.index.tolist()

    def get_model_name(self):
        return self.MODEL_NAME

    def _get_user_predictions(self, user_id):
        if user_id in self._prediction_cache:
            return self._prediction_cache[user_id]

        if user_id not in self.user_to_idx:
            return None

        u_idx = self.user_to_idx[user_id]
        user_vec = self.user_factors[u_idx]
        scores = user_vec @ self.item_factors.T
        self._prediction_cache[user_id] = scores
        return scores

    def recommend_items(self, user_id, items_to_ignore=None, topn=10, verbose=False):
        if items_to_ignore is None:
            items_to_ignore = []

        scores = self._get_user_predictions(user_id)
        if scores is None:
            candidate_items = [iid for iid in self.popular_items if iid not in items_to_ignore]
            top_items = candidate_items[:topn]
        else:
            item_scores = list(enumerate(scores))
            item_scores = [(idx, score) for idx, score in item_scores
                           if self.idx_to_item[idx] not in items_to_ignore]
            item_scores.sort(key=lambda x: x[1], reverse=True)
            top_indices = [idx for idx, _ in item_scores[:topn]]
            top_items = [self.idx_to_item[idx] for idx in top_indices]

        recs_df = pd.DataFrame(top_items, columns=['contentId'])
        if verbose and self.items_df is not None:
            recs_df = recs_df.merge(
                self.items_df[['contentId', 'title', 'url', 'lang']],
                on='contentId', how='left'
            )
        return recs_df

In [39]:
als_model = ALSRecommender(interactions_train_df, items_df=articles_df, n_factors=50)

print('Evaluating ALS recommendation model...')
als_global_metrics, als_detailed_results_df = model_evaluator.evaluate_model(als_model)
print('\nGlobal metrics:\n%s' % als_global_metrics)

Evaluating ALS recommendation model...
1140 users processed

Global metrics:
{'modelName': 'ALS (SVD)', 'recall@5': np.float64(0.3231909997443109), 'recall@10': np.float64(0.43594988493991305), 'ndcg@10': np.float64(0.26518300263783423)}


In [40]:
class SimpleHybrid:
    MODEL_NAME = 'SimpleHybrid'

    def __init__(self, content_model, als_model, topn_retrieve=200):
        self.content_model = content_model
        self.als_model = als_model
        self.topn_retrieve = topn_retrieve

    def get_model_name(self):
        return self.MODEL_NAME

    def recommend_items(self, user_id, items_to_ignore=None, topn=10, verbose=False):
        if items_to_ignore is None:
            items_to_ignore = []

        c_recs = self.content_model.recommend_items(user_id, items_to_ignore=[], topn=self.topn_retrieve, verbose=False)
        a_recs = self.als_model.recommend_items(user_id, items_to_ignore=[], topn=self.topn_retrieve, verbose=False)

        c_list = c_recs['contentId'].tolist() if not c_recs.empty else []
        a_list = a_recs['contentId'].tolist() if not a_recs.empty else []

        result = []
        i, j = 0, 0
        while len(result) < topn and (i < len(c_list) or j < len(a_list)):
            if i < len(c_list) and c_list[i] not in result and c_list[i] not in items_to_ignore:
                result.append(c_list[i])
            i += 1
            if len(result) >= topn:
                break
            if j < len(a_list) and a_list[j] not in result and a_list[j] not in items_to_ignore:
                result.append(a_list[j])
            j += 1

        if len(result) < topn:
            all_items = set(c_list + a_list)
            if hasattr(self.als_model, 'popular_items'):
                for item in self.als_model.popular_items:
                    if item not in result and item not in items_to_ignore:
                        result.append(item)
                        if len(result) == topn:
                            break

        recs_df = pd.DataFrame(result, columns=['contentId'])
        if verbose:
            if hasattr(self.content_model, 'items_df') and self.content_model.items_df is not None:
                items_df = self.content_model.items_df
                recs_df = recs_df.merge(items_df[['contentId', 'title', 'url', 'lang']], on='contentId', how='left')
        return recs_df

In [41]:
simple_hybrid = SimpleHybrid(content_model, als_model, topn_retrieve=200)
metrics = model_evaluator.evaluate_model(simple_hybrid)
print(metrics)

1140 users processed
({'modelName': 'SimpleHybrid', 'recall@5': np.float64(0.2582459728969573), 'recall@10': np.float64(0.39606238813602657), 'ndcg@10': np.float64(0.2179318709236983)},       hits@5_count  hits@10_count  interacted_count  recall@5  recall@10  \
76              15             37               192  0.078125   0.192708   
17              13             17               134  0.097015   0.126866   
16              12             24               130  0.092308   0.184615   
10              24             28               117  0.205128   0.239316   
82              10             28                88  0.113636   0.318182   
...            ...            ...               ...       ...        ...   
1094             0              0                 1  0.000000   0.000000   
1098             0              0                 1  0.000000   0.000000   
1099             0              0                 1  0.000000   0.000000   
1100             0              0                 1  0

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, edge_index):
        user_emb = self.user_embedding.weight
        item_emb = self.item_embedding.weight
        all_emb = torch.cat([user_emb, item_emb], dim=0)

        n_nodes = self.num_users + self.num_items
        adj = torch.zeros((n_nodes, n_nodes), device=edge_index.device)
        adj[edge_index[0], edge_index[1]] = 1.0

        deg = adj.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        adj_norm = deg_inv_sqrt.view(-1, 1) * adj * deg_inv_sqrt.view(1, -1)

        embeddings = [all_emb]
        for _ in range(self.num_layers):
            all_emb = torch.mm(adj_norm, all_emb)
            embeddings.append(all_emb)

        final_emb = torch.mean(torch.stack(embeddings, dim=0), dim=0)
        return final_emb


class LightGCNRanker:
    MODEL_NAME = 'LightGCN_Ranker'

    def __init__(self, interactions_train_df, items_df=None,
                 embedding_dim=64, num_layers=3, epochs=20,
                 lr=0.001, device=None):
        self.interactions_train_df = interactions_train_df
        self.items_df = items_df
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.epochs = epochs
        self.lr = lr
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        df = interactions_train_df.copy()
        df['personId'] = df['personId'].astype(str)
        df['contentId'] = df['contentId'].astype(str)

        unique_users = df['personId'].unique()
        unique_items = df['contentId'].unique()
        self.user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
        self.item_to_idx = {iid: j for j, iid in enumerate(unique_items)}
        self.idx_to_user = {i: uid for uid, i in self.user_to_idx.items()}
        self.idx_to_item = {j: iid for iid, j in self.item_to_idx.items()}

        self.num_users = len(self.user_to_idx)
        self.num_items = len(self.item_to_idx)
        print(f"Users: {self.num_users}, Items: {self.num_items}")

        user_indices = []
        item_indices = []
        for _, row in df.iterrows():
            u = self.user_to_idx[row['personId']]
            i = self.item_to_idx[row['contentId']]
            user_indices.append(u)
            item_indices.append(i)

        src = torch.tensor(user_indices + item_indices, dtype=torch.long)
        dst = torch.tensor(item_indices + user_indices, dtype=torch.long)
        self.edge_index = torch.stack([src, dst], dim=0).to(self.device)

        self.model = LightGCN(
            num_users=self.num_users,
            num_items=self.num_items,
            embedding_dim=self.embedding_dim,
            num_layers=self.num_layers
        ).to(self.device)

        from collections import defaultdict
        self.user_pos_items = defaultdict(list)
        for _, row in df.iterrows():
            u = self.user_to_idx[row['personId']]
            i = self.item_to_idx[row['contentId']]
            self.user_pos_items[u].append(i)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def get_model_name(self):
        return self.MODEL_NAME

    def train_model(self, verbose=True):
        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            users = list(self.user_pos_items.keys())
            np.random.shuffle(users)
            pbar = tqdm(users, desc=f"Epoch {epoch+1}/{self.epochs}", disable=not verbose)
            for u in pbar:
                pos = np.random.choice(self.user_pos_items[u])
                neg = np.random.randint(0, self.num_items)
                while neg in self.user_pos_items[u]:
                    neg = np.random.randint(0, self.num_items)

                u_t = torch.tensor([u], device=self.device)
                pos_t = torch.tensor([pos], device=self.device)
                neg_t = torch.tensor([neg], device=self.device)

                self.optimizer.zero_grad()
                all_emb = self.model(self.edge_index)
                user_emb = all_emb[u_t]
                pos_emb = all_emb[self.num_users + pos_t]
                neg_emb = all_emb[self.num_users + neg_t]

                pos_score = (user_emb * pos_emb).sum(dim=1)
                neg_score = (user_emb * neg_emb).sum(dim=1)
                loss = -torch.log(torch.sigmoid(pos_score - neg_score)).mean()
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})

            if verbose:
                print(f"Epoch {epoch+1} average loss: {total_loss/len(users):.4f}")

        self.model.eval()
        with torch.no_grad():
            final_emb = self.model(self.edge_index)
            self.user_emb = final_emb[:self.num_users].cpu().numpy()
            self.item_emb = final_emb[self.num_users:].cpu().numpy()

    def recommend_items(self, user_id, items_to_ignore=None, topn=10, candidate_items=None, verbose=False):
        if items_to_ignore is None:
            items_to_ignore = []
        user_id_str = str(int(user_id))
        items_to_ignore_str = [str(int(x)) for x in items_to_ignore]

        if user_id_str not in self.user_to_idx:
            pop = self.interactions_train_df.groupby('contentId').size().sort_values(ascending=False).index
            pop_str = [str(int(x)) for x in pop]
            recs = [i for i in pop_str if i not in items_to_ignore_str][:topn]
            recs_int = [int(x) for x in recs]
            return pd.DataFrame(recs_int, columns=['contentId'])

        u_idx = self.user_to_idx[user_id_str]
        user_vec = self.user_emb[u_idx]

        if candidate_items is None:
            item_indices = list(range(self.num_items))
            item_ids = [self.idx_to_item[i] for i in item_indices]
        else:
            cand_str = [str(int(x)) for x in candidate_items]
            item_ids = [iid for iid in cand_str if iid in self.item_to_idx]
            item_indices = [self.item_to_idx[iid] for iid in item_ids]

        if not item_indices:
            return pd.DataFrame(columns=['contentId'])

        item_vecs = self.item_emb[item_indices]
        scores = np.dot(item_vecs, user_vec)

        sorted_pairs = sorted(zip(item_ids, scores), key=lambda x: x[1], reverse=True)
        recommendations = [iid for iid, _ in sorted_pairs if iid not in items_to_ignore_str][:topn]
        recommendations_int = [int(x) for x in recommendations]

        recs_df = pd.DataFrame(recommendations_int, columns=['contentId'])
        if verbose and self.items_df is not None:
            recs_df = recs_df.merge(
                self.items_df[['contentId', 'title', 'url', 'lang']],
                on='contentId', how='left'
            )
        return recs_df

In [43]:
retriever = SimpleHybrid(content_model, als_model, topn_retrieve=200)

ranker = LightGCNRanker(interactions_train_df, items_df=articles_df, embedding_dim=64, epochs=20)
ranker.train_model()

class TwoStageRecommender:
    MODEL_NAME = 'TwoStage'
    def __init__(self, retriever, ranker):
        self.retriever = retriever
        self.ranker = ranker

    def get_model_name(self):
        return self.MODEL_NAME

    def recommend_items(self, user_id, items_to_ignore=None, topn=10, verbose=False):
        candidates_df = self.retriever.recommend_items(user_id, items_to_ignore=items_to_ignore, topn=200, verbose=False)
        candidate_items = candidates_df['contentId'].tolist()
        ranked = self.ranker.recommend_items(user_id, items_to_ignore=items_to_ignore, topn=topn,
                                             candidate_items=candidate_items, verbose=verbose)
        return ranked

two_stage = TwoStageRecommender(retriever=simple_hybrid, ranker=ranker)
metrics = model_evaluator.evaluate_model(two_stage)
print(metrics)

Users: 1140, Items: 2926


Epoch 1/20: 100%|██████████| 1140/1140 [00:07<00:00, 142.56it/s, loss=0.6150]


Epoch 1 average loss: 0.6858


Epoch 2/20: 100%|██████████| 1140/1140 [00:07<00:00, 142.66it/s, loss=0.8110]


Epoch 2 average loss: 0.6530


...

Epoch 18 average loss: 0.5209


Epoch 19/20: 100%|██████████| 1140/1140 [00:08<00:00, 141.45it/s, loss=0.2260]


Epoch 19 average loss: 0.4569


Epoch 20/20: 100%|██████████| 1140/1140 [00:08<00:00, 141.07it/s, loss=0.5793]


Epoch 20 average loss: 0.4634
1140 users processed
({'modelName': 'TwoStage', 'recall@5': np.float64(0.2717974942469957), 'recall@10': np.float64(0.3139861927895679), 'ndcg@10': np.float64(0.20111758898653825)},       hits@5_count  hits@10_count  interacted_count  recall@5  recall@10  \
76              23             29               192  0.119792   0.151042   
17               9             14               134  0.067164   0.104478   
16              13             19               130  0.100000   0.146154   
10               8             25               117  0.068376   0.213675   
82              17             18                88  0.193182   0.204545   
...            ...            ...               ...       ...        ...   
1094             0              0                 1  0.000000   0.000000   
1098             0              0                 1  0.000000   0.000000   
1099             0              0                 1  0.000000   0.000000   
1100             0          

In [44]:
import joblib

model_state = {

    'als': {
        'user_factors': als_model.user_factors,
        'item_factors': als_model.item_factors,
        'user_to_idx': als_model.user_to_idx,
        'item_to_idx': als_model.item_to_idx,
        'idx_to_item': als_model.idx_to_item,
        'popular_items': als_model.popular_items,
    },

    'ranker': {
        'user_emb': ranker.user_emb,
        'item_emb': ranker.item_emb,
        'user_to_idx': ranker.user_to_idx,
        'item_to_idx': ranker.item_to_idx,
        'idx_to_item': ranker.idx_to_item,
    }
}

joblib.dump(model_state, 'model_state.joblib')
articles_df.to_csv('articles.csv', index=False)